# 03 — Ollama + Gemma на GPU (рекомендуемый)

Самый простой способ запустить полноценную LLM на GPU в Colab. Ollama предоставляет нативный OpenAI-совместимый endpoint.

**Требования:** Google Colab с GPU (T4/V100)  
**GPU:** ✅ Обязательно  
**API ключ:** Не нужен  
**Модели:** gemma3:4b (~3GB), gemma3:12b (~8GB)

**Кэш модели:** Модель кэшируется на Google Drive — повторный запуск не качает заново!

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu}")
    print(f"VRAM: {mem:.1f} GB")
    MODEL = "gemma3:4b" if mem < 12 else "gemma3:12b"
    print(f"\nРекомендуемая модель: {MODEL}")
else:
    print("⚠ GPU не доступен! Перейдите: Runtime → Change runtime type → T4 GPU")
    MODEL = "gemma3:4b"

In [ ]:
!sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.ai/install.sh | sh

# Проверка установки
import shutil
if shutil.which("ollama"):
    print("✓ Ollama установлен")
else:
    print("✗ Ollama не установлен! Попробуйте перезапустить ячейку")

In [ ]:
import subprocess, time, os, shutil

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"

# Start ollama serve in background
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)

# --- Google Drive кэш модели ---
DRIVE_CACHE = "/content/drive/MyDrive/ollama_cache"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
os.makedirs(DRIVE_CACHE, exist_ok=True)

# Убеждаемся что zstd доступен (установлен в предыдущей ячейке)
subprocess.run(["which", "zstd"], capture_output=True).returncode == 0 or \
    subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], capture_output=True)

cache_file = f"{DRIVE_CACHE}/{MODEL.replace(':', '_')}.tar.zst"
ollama_dir = os.path.expanduser("~/.ollama/models")

restored = False

if os.path.exists(cache_file):
    size_gb = os.path.getsize(cache_file) / 1024**3
    print(f"Восстановление {MODEL} из Google Drive кэша ({size_gb:.1f} GB)...")
    t0 = time.time()
    os.makedirs(ollama_dir, exist_ok=True)

    # Распаковка: zstd -d -> tar x (надёжнее чем tar -I zstd)
    ret = subprocess.run(
        f"zstd -d '{cache_file}' --stdout | tar xf - -C ~/.ollama/models",
        shell=True
    )

    if ret.returncode == 0:
        elapsed = time.time() - t0
        # Проверяем что ollama видит модель
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
        if MODEL.split(":")[0] in result.stdout:
            print(f"✓ Восстановлено за {elapsed:.0f}с")
            print(f"✓ Модель {MODEL} готова (из кэша)")
            restored = True
        else:
            print(f"⚠ Кэш битый (модель не видна), удаляю...")
            os.remove(cache_file)
    else:
        print(f"⚠ Ошибка распаковки (битый архив?), удаляю кэш...")
        os.remove(cache_file)

if not restored:
    # Скачиваем и кэшируем
    print(f"Скачивание модели {MODEL}...")
    subprocess.run(["ollama", "pull", MODEL], check=True)
    print(f"✓ Модель {MODEL} скачана")

    print(f"Сохранение в Google Drive кэш...")
    t0 = time.time()
    # Сжатие: tar -> zstd (pipe, надёжнее)
    ret = subprocess.run(
        f"tar cf - -C ~/.ollama/models . | zstd -o '{cache_file}'",
        shell=True
    )
    if ret.returncode == 0:
        size_gb = os.path.getsize(cache_file) / 1024**3
        elapsed = time.time() - t0
        print(f"✓ Кэш сохранён: {size_gb:.1f} GB за {elapsed:.0f}с")
    else:
        print(f"⚠ Не удалось сохранить кэш (не критично)")

print(f"\n✓ Модель {MODEL} готова к работе")

In [ ]:
import requests, json

response = requests.post(
    "http://localhost:11434/v1/chat/completions",
    json={
        "model": MODEL,
        "messages": [{"role": "user", "content": "Say hello in one word"}],
        "temperature": 0.0,
        "max_tokens": 10
    }
)

if response.status_code == 200:
    data = response.json()
    print(f"✓ Модель отвечает: {data['choices'][0]['message']['content']}")
else:
    print(f"✗ Ошибка: {response.status_code} {response.text}")

In [ ]:
!pip install -q fastapi uvicorn httpx

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import httpx, uvicorn, threading, time, uuid, socket

app = FastAPI(title="Ollama Proxy")
OLLAMA_URL = "http://localhost:11434"

@app.get("/health")
async def health():
    try:
        async with httpx.AsyncClient() as client:
            r = await client.get(f"{OLLAMA_URL}/api/tags", timeout=5)
        return {"status": "ok", "model": MODEL, "backend": "ollama"}
    except:
        return JSONResponse(status_code=503, content={"status": "error"})

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    if body.get("model") in ("local", "default", "gpt-3.5-turbo", "gpt-4"):
        body["model"] = MODEL
    body["stream"] = False

    try:
        async with httpx.AsyncClient(timeout=120) as client:
            r = await client.post(f"{OLLAMA_URL}/v1/chat/completions", json=body)
        return JSONResponse(content=r.json(), status_code=r.status_code)
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"error": {"message": str(e), "type": "server_error"}}
        )

# Автоматический выбор свободного порта
for _port in [8080, 8081, 8082, 9090]:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(('localhost', _port)) != 0:
            PORT = _port
            break
else:
    with socket.socket() as s:
        s.bind(('', 0))
        PORT = s.getsockname()[1]

def run_proxy():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

thread = threading.Thread(target=run_proxy, daemon=True)
thread.start()
time.sleep(2)
print(f"✓ Прокси запущен на порту {PORT} → Ollama :11434")

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess, re, time

process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

tunnel_url = None
for _ in range(30):
    line = process.stderr.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(1)

if tunnel_url:
    print(f"✓ Туннель активен!")
    print(f"\n{'='*60}")
    print(f"  URL: {tunnel_url}")
    print(f"  Health: {tunnel_url}/health")
    print(f"  API: {tunnel_url}/v1/chat/completions")
    print(f"{'='*60}")
else:
    print("✗ Не удалось получить URL туннеля")

In [ ]:
!pip install -q openai

from openai import OpenAI

client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="not-needed")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": """Extract from the invoice text: country, doc_type (electricity/telecom/bank/water/gas/tax/other), company, year.
Respond ONLY with JSON: {"country": "...", "doc_type": "...", "company": "...", "year": ...}"""},
        {"role": "user", "content": """RECHNUNG
Wiener Netze GmbH
Erdbergstraße 236, 1110 Wien
Rechnungsdatum: 15.03.2024
Stromrechnung für den Zeitraum 01.01.2024 - 31.03.2024
Gesamtbetrag: EUR 245,67 inkl. MwSt
UID: ATU12345678"""}
    ],
    temperature=0.0,
    max_tokens=200
)

print("Ответ модели:")
print(response.choices[0].message.content)

## Подключение к InvoiceLLM

Добавьте в `config.yaml`:

```yaml
llm:
  servers:
    - name: "Colab Ollama"
      host: "YOUR-URL.trycloudflare.com"
      port: 443
      ssl: true
      priority: 1
```

### Смена модели

```python
# Скачать другую модель
!ollama pull llama3.2:3b
!ollama pull gemma3:12b

# Список установленных
!ollama list
```